# Here I try to use the Invertible network to convert matrix to a set of gates

In [13]:
import torch
import torch.nn as nn
import FrEIA.framework as Ff
import FrEIA.modules as Fm

import numpy as np

from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
import torch.optim as optim

import qiskit
from qiskit.quantum_info import Operator

In [4]:
#!pip install FrEIA

In [127]:
# -------------------------
# Define subnet constructor
# -------------------------
def subnet_fc(c_in, c_out):
    return nn.Sequential(
        nn.Linear(c_in, 128),
        nn.ReLU(),
        nn.Linear(128, c_out)
    )

# -------------------------
# Define INN architecture
# -------------------------
def build_inn(input_dim=18, output_dim=32, num_blocks=6):
    nodes = [Ff.InputNode(output_dim, name='input')]

    for i in range(num_blocks):
        nodes.append(Ff.Node(nodes[-1],
                             Fm.GLOWCouplingBlock,
                             {'subnet_constructor': subnet_fc, 'clamp': 2.0},
                             name=f'coupling_{i}'))
        nodes.append(Ff.Node(nodes[-1],
                             Fm.PermuteRandom,
                             {'seed': i},
                             name=f'permute_{i}'))

    nodes.append(Ff.OutputNode(nodes[-1], name='output'))
    return Ff.ReversibleGraphNet(nodes, verbose=False)

# -------------------------
# Loss Functions
# -------------------------
def forward_loss(pred_op, target_op):
    return nn.MSELoss()(pred_op, target_op)

def backward_loss(pred_angles, target_angles):
    return nn.MSELoss()(pred_angles, target_angles)

# -------------------------
# Example Training Step
# -------------------------
def train_step(model, optimizer, angles, operators):
    model.train()
    noise = torch.randn_like(angles[:, :14])  # 14D latent
    input_forward = torch.cat([angles, noise], dim=1)  # shape (batch, 32)

    pred_operator, _ = model(input_forward)  # unpack output tensor and ignore logdet
    loss_fwd = forward_loss(pred_operator, operators)

    recovered, _ = model(operators, rev=True)
    pred_angles = recovered[:, :18]
    loss_bwd = backward_loss(pred_angles, angles)

    loss = loss_fwd + loss_bwd
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item(), loss_fwd.item(), loss_bwd.item()
# -------------------------
# Full Training Loop
# -------------------------
def train_inn(model, optimizer, dataloader, epochs=100):
    for epoch in range(epochs):
        total_loss = 0
        for angles_batch, op_batch in dataloader:
            loss, lf, lb = train_step(model, optimizer, angles_batch, op_batch)
            total_loss += loss
        print(f"Epoch {epoch+1}: Total Loss = {total_loss:.4f}")


In [128]:
def get_matrix_for_double_cz_block(angles):
    qc = qiskit.QuantumCircuit(2)
    qc.u(angles[0], angles[1], angles[2], 0)
    qc.u(angles[3], angles[4], angles[5], 1)
    qc.cz(0, 1)
    qc.u(angles[6], angles[7], angles[8], 0)
    qc.u(angles[9], angles[10], angles[11], 1)
    qc.cz(0, 1)
    qc.u(angles[12], angles[13], angles[14], 0)
    qc.u(angles[15], angles[16], angles[17], 1)
    op = Operator(qc)
    matrix = op.data
    return matrix

In [132]:
def generate_dataset(N=100000):
    X = []  # main_operator (flattened)
    Y = []  # angles
    for _ in range(N):
        angles = np.random.uniform(0, 2*np.pi, size=18)
        main_operator = get_matrix_for_double_cz_block(angles)  
        Y.append(np.concatenate([main_operator.real.flatten(), main_operator.imag.flatten()]))
        X.append(angles)
    
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)


In [133]:
# Generate data
X, Y = generate_dataset()
dataset = TensorDataset(X, Y)

print(loader)


In [135]:
torch.manual_seed(0)
loader = DataLoader(dataset, batch_size=100000, shuffle=True)

# Initialize model and optimizer
model = build_inn()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train the INN
train_inn(model, optimizer, loader, epochs=200)

Epoch 1: Total Loss = 469596.9062
Epoch 2: Total Loss = 10365.7246
Epoch 3: Total Loss = 596.4440
Epoch 4: Total Loss = 98.9377
Epoch 5: Total Loss = 31.2676
Epoch 6: Total Loss = 21.7226
Epoch 7: Total Loss = 18.3086
Epoch 8: Total Loss = 16.8885
Epoch 9: Total Loss = 15.8850
Epoch 10: Total Loss = 15.4057
Epoch 11: Total Loss = 14.9531
Epoch 12: Total Loss = 14.6195
Epoch 13: Total Loss = 14.5245
Epoch 14: Total Loss = 14.1443
Epoch 15: Total Loss = 13.9693
Epoch 16: Total Loss = 13.8171
Epoch 17: Total Loss = 13.7927
Epoch 18: Total Loss = 13.5689
Epoch 19: Total Loss = 13.5837
Epoch 20: Total Loss = 13.4644
Epoch 21: Total Loss = 13.3965
Epoch 22: Total Loss = 13.3486
Epoch 23: Total Loss = 13.3317
Epoch 24: Total Loss = 13.3051
Epoch 25: Total Loss = 13.2877
Epoch 26: Total Loss = 13.2647
Epoch 27: Total Loss = 13.2632
Epoch 28: Total Loss = 13.2632
Epoch 29: Total Loss = 13.2714
Epoch 30: Total Loss = 13.2367
Epoch 31: Total Loss = 13.2467
Epoch 32: Total Loss = 13.2666
Epoch 33:

In [136]:
def test_inn(model, test_angles, test_operators, n_samples=2):
    model.eval()

    with torch.no_grad():
        # Forward pass: predict operators from angles + noise
        noise = torch.randn(test_angles.shape[0], 14)
        input_tensor = torch.cat([test_angles, noise], dim=1)
        pred_ops = model(input_tensor)

        # Backward pass: recover angles from true operators
        recovered = model(test_operators, rev=True)
        #print(recovered[0])
        pred_angles = recovered[0][:, :18]

        for i in range(n_samples):
            print(f"\nSample {i + 1}:")
            print("Original Angles: ", test_angles[i].numpy())
            print("Predicted Angles:", pred_angles[i].numpy())
            print("Target Operator: ", test_operators[i].numpy())
            print("Predicted Operator:", pred_ops[0][i].numpy())

In [137]:
# Create angle/operator pairs
angles = []
operators = []

for _ in range(5):  # generate 5 test samples
    ang = np.random.uniform(0, 2 * np.pi, size=18)
    angles.append(ang)
    main_operator = get_matrix_for_double_cz_block(ang)  # your function
    op = np.concatenate([main_operator.real.flatten(), main_operator.imag.flatten()])
    operators.append(op)

# Convert to torch tensors
angles = torch.tensor(angles, dtype=torch.float32)
operators = torch.tensor(operators, dtype=torch.float32)

# Test the model
test_inn(model, angles, operators, n_samples=5)


Sample 1:
Original Angles:  [4.5984817  0.09161356 0.58661723 5.1933937  5.2369895  5.607184
 6.0193763  3.5269852  0.56893426 6.2579827  2.9996035  4.3034515
 5.299083   3.8712265  3.5386114  2.3132312  4.3412485  5.128133  ]
Predicted Angles: [ 0.94825894 -0.30867687  3.2710476  -0.1493801   0.9892348   2.055932
  2.0208647   2.560006    0.19684745  1.5735495   0.81833774  0.2779105
  1.9739294   1.3121444   0.70635366  2.685899    1.9034215   1.3645692 ]
Target Operator:  [-0.14650115 -0.4816086  -0.35081333  0.21999481 -0.24543728 -0.3177075
  0.484829   -0.26523775  0.30577192 -0.4427129   0.39816138  0.2454908
 -0.571575   -0.15857211 -0.13382527  0.37422442  0.46792847 -0.17562163
 -0.29513037 -0.4880788  -0.46064216  0.2237807  -0.207233   -0.4777004
  0.2570924   0.3711944  -0.28480336  0.45466018 -0.02930444  0.47672763
  0.50425065  0.08835761]
Predicted Operator: [-0.470871    1.2984004  -0.96536267 -0.2258914  -0.9976388   1.3273623
  0.05825248 -0.17925549 -1.0014304   0